# Bayesian Optimization of a PyTorch Model Using JAXBO

This notebook demonstrates how to use [JAXBO](https://github.com/ricardogr07/JAX-BO) to optimize the hyperparameters of a simple PyTorch neural network. The objective is to approximate a simple function \( z = x^2 + y^2 \), and use Bayesian Optimization to minimize the MSE on a validation set.

We'll optimize the learning rate, weight decay, and number of epochs of training.

## Setup
First, ensure the following packages are installed in your environment:
- torch
- jax
- jaxlib
- jaxbo
- matplotlib
- scikit-learn

In [ ]:
!pip install torch jax jaxlib matplotlib scikit-learn jaxbo

In [2]:
# Imports
import json
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import jax.numpy as jnp
from jax import random

from jaxbo.models import GP
from jaxbo.input_priors import uniform_prior
from jaxbo.utils import normalize

## Step 1: Define the Dataset and PyTorch Model

We generate a synthetic dataset where the true function is \( z = x^2 + y^2 \), and define a simple 2-layer MLP in PyTorch to fit it.

In [3]:
# Generate synthetic dataset
def generate_dataset(n_samples=1000, seed=0):
    torch.manual_seed(seed)
    X = 4 * torch.rand(n_samples, 2) - 2  # range [-2, 2]
    y = torch.sum(X ** 2, dim=1, keepdim=True)
    return X.numpy(), y.numpy()

X, y = generate_dataset()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Define a simple PyTorch model
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

## Step 2: Define the Black-Box Objective

This function trains the model using given hyperparameters and returns the MSE on the validation set. We will optimize:
- `lr` (learning rate)
- `weight_decay`
- `epochs`

In [5]:
def train_and_evaluate_model(lr=1e-3, weight_decay=1e-4, epochs=100):
    model = SimpleMLP()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_t)
        loss = criterion(y_pred, y_t)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    X_v = torch.tensor(X_val, dtype=torch.float32)
    y_v = torch.tensor(y_val, dtype=torch.float32)
    with torch.no_grad():
        y_pred_val = model(X_v)
        val_loss = criterion(y_pred_val, y_v).item()
    return val_loss

## Step 3: Setup Bayesian Optimization with JAXBO


In [6]:
# Search space bounds
lb = jnp.array([1e-4, 0.0, 10.0])      # lr, weight_decay, epochs
ub = jnp.array([1e-1, 0.1, 1000.0])
bounds = {'lb': lb, 'ub': ub}
dim = 3
p_x = uniform_prior(lb=lb, ub=ub)

def black_box_objective(x: jnp.ndarray) -> jnp.ndarray:
    lr = float(x[0])
    weight_decay = float(x[1])
    epochs = int(jnp.round(x[2]))
    val_loss = train_and_evaluate_model(lr, weight_decay, epochs)
    return jnp.array(val_loss)

## Step 4: Initial Sampling and Optimization Loop


In [7]:
options = {
    'kernel': 'RBF',
    'criterion': 'LCB',
    'input_prior': p_x,
    'kappa': 2.0,
    'nIter': 15
}

model = GP(options)
rng_key = random.PRNGKey(0)

def sample_initial_data(fn, lb, ub, dim, n_samples, rng_key):
    X_init = lb + (ub - lb) * random.uniform(rng_key, shape=(n_samples, dim))
    y_init = jnp.array([fn(x) for x in X_init])
    return X_init, y_init

X_init, y_init = sample_initial_data(black_box_objective, lb, ub, dim, 5, rng_key)
X, y = X_init, y_init

In [8]:
# Optimization loop
for it in range(options['nIter']):
    print(f"\nIteration {it+1}/{options['nIter']}")

    norm_batch, norm_const = normalize(X, y, bounds)

    rng_key, subkey = random.split(rng_key)
    opt_params = model.train(norm_batch, subkey, num_restarts=5)

    new_X, _, _ = model.compute_next_point_lbfgs(
        num_restarts=5,
        params=opt_params,
        batch=norm_batch,
        norm_const=norm_const,
        bounds=bounds,
        kappa=options['kappa'],
        gmm_vars=None,
        rng_key=subkey
    )

    new_y = jnp.array([black_box_objective(x) for x in new_X])
    X = jnp.vstack([X, new_X])
    y = jnp.hstack([y, new_y])

    best_idx = jnp.argmin(y)
    best_x = X[best_idx]
    best_y = y[best_idx]

    print(f"Best so far: lr={float(best_x[0]):.5f}, wd={float(best_x[1]):.5f}, epochs={int(jnp.round(best_x[2]))}")
    print(f"Val loss: {float(best_y):.6f}")


Iteration 1/15
Best so far: lr=0.10000, wd=0.02464, epochs=710
Val loss: 0.060352

Iteration 2/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 3/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 4/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 5/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 6/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 7/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 8/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 9/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 10/15
Best so far: lr=0.06687, wd=0.00259, epochs=556
Val loss: 0.005983

Iteration 11/15
Best so far: lr=0.01725, wd=0.00000, epochs=875
Val loss: 0.004009

Iteration 12/15
Best so far: lr=0.01725, wd=0.00000, epochs=875
Val loss: 0.004009



## Step 5: Final Results


In [10]:
best_idx = jnp.argmin(y)
best_x = X[best_idx]
best_y = y[best_idx]

best_hparams = {
    "lr": float(best_x[0]),
    "weight_decay": float(best_x[1]),
    "epochs": int(jnp.round(best_x[2]))
}

print("\nBest Hyperparameters Found:")
print(json.dumps(best_hparams, indent=4))
print(f"Best validation loss: {float(best_y):.6f}")


Best Hyperparameters Found:
{
    "lr": 0.017246901988983154,
    "weight_decay": 0.0,
    "epochs": 875
}
Best validation loss: 0.004009
